In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform


returns_df = pd.read_excel("daily_returns.xlsx")
categories_df = pd.read_excel("stock_categories.xlsx")

# Using full dataset (3 years)
print(f"Data loaded: {len(returns_df)} days total")
print(f"Date range: {returns_df['Date'].iloc[0]} to {returns_df['Date'].iloc[-1]}")
print()

sectors = categories_df.groupby("Category")["Symbol"].apply(list).to_dict()
corr_matrix_full = returns_df.drop(columns=["Date"]).corr()

n_stocks = len(corr_matrix_full)
avg_corr_per_stock = (corr_matrix_full.sum(axis=0) - 1.0) / (n_stocks - 1)
avg_corr_per_stock = avg_corr_per_stock.sort_values()
print("Average correlation for each stock (with all other stocks):")
print(avg_corr_per_stock.to_string())
print()

risk_free_rate = 0.065  # 6.5% Annual
trading_days = 252

# Calculate Sharpe ratio for each stock (needed for max Sharpe selection)
stock_returns = returns_df.drop(columns=["Date"])
mean_returns = stock_returns.mean() / 100 * trading_days  # Annualized
std_returns = stock_returns.std() / 100 * np.sqrt(trading_days)  # Annualized
sharpe_ratios = (mean_returns - risk_free_rate) / std_returns

# ============================================================
# METHOD 1: Hierarchical Risk Parity (HRP) Selection
# ============================================================
print("=" * 70)
print("METHOD 1: Hierarchical Risk Parity (HRP) Stock Selection")
print("=" * 70)

hrp_stocks = []
for sector, stocks in sectors.items():
    if len(stocks) == 1:
        hrp_stocks.append(stocks[0])
        print(f"{sector}: {stocks[0]} (only stock in sector)")
    else:
        sector_corr = corr_matrix_full.loc[stocks, stocks]
        dist_matrix = np.sqrt(0.5 * (1 - sector_corr))
        dist_condensed = squareform(dist_matrix, checks=False)
        linkage_matrix = linkage(dist_condensed, method='ward')
        avg_distances = dist_matrix.sum(axis=0) / (len(stocks) - 1)
        best_stock = avg_distances.idxmax()
        hrp_stocks.append(best_stock)
        print(f"{sector}: {best_stock} (avg distance: {avg_distances[best_stock]:.4f})")

print(f"\nSelected {len(hrp_stocks)} stocks using HRP")
print()

# ============================================================
# METHOD 2: Maximum Sharpe Ratio Selection
# ============================================================
print("=" * 70)
print("METHOD 2: Maximum Sharpe Ratio Stock Selection")
print("=" * 70)

sharpe_stocks = []
for sector, stocks in sectors.items():
    sector_sharpe = sharpe_ratios[stocks]
    best_stock = sector_sharpe.idxmax()
    sharpe_stocks.append(best_stock)
    print(f"{sector}: {best_stock} (Sharpe: {sharpe_ratios[best_stock]:.4f})")

print(f"\nSelected {len(sharpe_stocks)} stocks using Max Sharpe")
print()

# ============================================================
# Compare Stock Selections
# ============================================================
print("=" * 70)
print("COMPARING STOCK SELECTIONS")
print("=" * 70)
comparison_df = pd.DataFrame({
    "Sector": list(sectors.keys()),
    "HRP Selection": hrp_stocks,
    "Sharpe Selection": sharpe_stocks,
    "Same?": ["✓" if h == s else "✗" for h, s in zip(hrp_stocks, sharpe_stocks)]
})
print(comparison_df.to_string(index=False))
overlap = sum(h == s for h, s in zip(hrp_stocks, sharpe_stocks))
print(f"\nOverlap: {overlap}/{len(hrp_stocks)} stocks are the same")
print()

# ============================================================
# Portfolio Optimization for BOTH Methods
# ============================================================

def optimize_portfolio(selected_stocks, stock_returns_df, method_name):
    """Run portfolio optimization and return results"""
    print(f"\n{'='*70}")
    print(f"PORTFOLIO OPTIMIZATION: {method_name}")
    print(f"{'='*70}")
    
    selected_returns = stock_returns_df[selected_stocks]
    
    mean_daily_returns = selected_returns.mean() / 100
    cov_matrix_daily = selected_returns.cov() / 10000
    
    annual_returns = mean_daily_returns * trading_days
    annual_cov = cov_matrix_daily * trading_days
    
    np.random.seed(42)
    num_portfolios = 10000
    results = np.zeros((3, num_portfolios))
    weights_record = []
    
    for i in range(num_portfolios):
        weights = np.random.random(len(selected_stocks))
        weights /= np.sum(weights)
        weights_record.append(weights)
        
        portfolio_return = np.dot(weights, annual_returns)
        portfolio_std_dev = np.sqrt(np.dot(weights.T, np.dot(annual_cov, weights)))
        
        results[0, i] = portfolio_return
        results[1, i] = portfolio_std_dev
        results[2, i] = (portfolio_return - risk_free_rate) / portfolio_std_dev
    
    max_sharpe_idx = np.argmax(results[2])
    max_sharpe_w = weights_record[max_sharpe_idx]
    
    min_vol_idx = np.argmin(results[1])
    min_vol_w = weights_record[min_vol_idx]
    
    weights_table = pd.DataFrame({
        "Stock": selected_stocks,
        "GMV weight": min_vol_w,
        "Max Sharpe weight": max_sharpe_w,
    })
    weights_table["GMV weight"] = weights_table["GMV weight"].map(lambda x: f"{x:.2%}")
    weights_table["Max Sharpe weight"] = weights_table["Max Sharpe weight"].map(lambda x: f"{x:.2%}")
    print("\nWeights: Global Minimum Variance (GMV) vs Maximum Sharpe")
    print(weights_table.to_string(index=False))
    print()
    
    perf_table = pd.DataFrame({
        "Portfolio": ["GMV (min variance)", "Max Sharpe"],
        "Return (ann.)": [results[0, min_vol_idx], results[0, max_sharpe_idx]],
        "Volatility (ann.)": [results[1, min_vol_idx], results[1, max_sharpe_idx]],
        "Sharpe Ratio": [results[2, min_vol_idx], results[2, max_sharpe_idx]],
    })
    perf_table["Return (ann.)"] = perf_table["Return (ann.)"].map(lambda x: f"{x:.2%}")
    perf_table["Volatility (ann.)"] = perf_table["Volatility (ann.)"].map(lambda x: f"{x:.2%}")
    perf_table["Sharpe Ratio"] = perf_table["Sharpe Ratio"].map(lambda x: f"{x:.4f}")
    print("Return, Volatility, and Sharpe Ratio (annualized):")
    print(perf_table.to_string(index=False))
    print()
    
    return results, selected_returns, max_sharpe_idx, min_vol_idx, min_vol_w, max_sharpe_w

# Run optimization for both methods
hrp_results, hrp_returns, hrp_max_sharpe_idx, hrp_min_vol_idx, hrp_gmv_w, hrp_sharpe_w = optimize_portfolio(
    hrp_stocks, returns_df, "HRP Selection"
)

sharpe_results, sharpe_returns, sharpe_max_sharpe_idx, sharpe_min_vol_idx, sharpe_gmv_w, sharpe_sharpe_w = optimize_portfolio(
    sharpe_stocks, returns_df, "Max Sharpe Selection"
)
# ============================================================
# FINAL COMPARISON
# ============================================================
print("\n" + "=" * 70)
print("FINAL PERFORMANCE COMPARISON")
print("=" * 70)
final_comparison = pd.DataFrame({
    "Method": ["HRP - GMV", "HRP - Max Sharpe", "Sharpe - GMV", "Sharpe - Max Sharpe"],
    "Return": [
        hrp_results[0, hrp_min_vol_idx],
        hrp_results[0, hrp_max_sharpe_idx],
        sharpe_results[0, sharpe_min_vol_idx],
        sharpe_results[0, sharpe_max_sharpe_idx]
    ],
    "Volatility": [
        hrp_results[1, hrp_min_vol_idx],
        hrp_results[1, hrp_max_sharpe_idx],
        sharpe_results[1, sharpe_min_vol_idx],
        sharpe_results[1, sharpe_max_sharpe_idx]
    ],
    "Sharpe": [
        hrp_results[2, hrp_min_vol_idx],
        hrp_results[2, hrp_max_sharpe_idx],
        sharpe_results[2, sharpe_min_vol_idx],
        sharpe_results[2, sharpe_max_sharpe_idx]
    ]
})
final_comparison["Return"] = final_comparison["Return"].map(lambda x: f"{x:.2%}")
final_comparison["Volatility"] = final_comparison["Volatility"].map(lambda x: f"{x:.2%}")
final_comparison["Sharpe"] = final_comparison["Sharpe"].map(lambda x: f"{x:.4f}")
print(final_comparison.to_string(index=False))
print()

# ============================================================
# Visualizations
# ============================================================
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.size"] = 11
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

# Figure 1: Side-by-side Efficient Frontiers
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), facecolor="#fafafa")

# HRP Plot
ax1.set_facecolor("white")
sc1 = ax1.scatter(
    hrp_results[1, :],
    hrp_results[0, :],
    c=hrp_results[2, :],
    cmap="viridis",
    s=12,
    alpha=0.5,
    edgecolors="none",
)
cbar1 = plt.colorbar(sc1, ax=ax1, shrink=0.8, pad=0.02)
cbar1.set_label("Sharpe Ratio", fontweight="medium")
cbar1.ax.tick_params(labelsize=9)

ax1.scatter(
    hrp_results[1, hrp_max_sharpe_idx],
    hrp_results[0, hrp_max_sharpe_idx],
    marker="*",
    color="#dc2626",
    s=300,
    edgecolors="white",
    linewidths=1.5,
    label="Max Sharpe",
    zorder=5,
)
ax1.scatter(
    hrp_results[1, hrp_min_vol_idx],
    hrp_results[0, hrp_min_vol_idx],
    marker="D",
    color="#059669",
    s=100,
    edgecolors="white",
    linewidths=1.5,
    label="GMV",
    zorder=5,
)

ax1.set_xlabel("Annualized Volatility", fontweight="medium")
ax1.set_ylabel("Annualized Return", fontweight="medium")
ax1.set_title("Efficient Frontier: HRP Selection", fontweight="600", pad=12)
ax1.legend(loc="upper right", frameon=True)
ax1.grid(True, linestyle="--", alpha=0.5, color="#94a3b8")
ax1.set_axisbelow(True)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# Max Sharpe Plot
ax2.set_facecolor("white")
sc2 = ax2.scatter(
    sharpe_results[1, :],
    sharpe_results[0, :],
    c=sharpe_results[2, :],
    cmap="viridis",
    s=12,
    alpha=0.5,
    edgecolors="none",
)
cbar2 = plt.colorbar(sc2, ax=ax2, shrink=0.8, pad=0.02)
cbar2.set_label("Sharpe Ratio", fontweight="medium")
cbar2.ax.tick_params(labelsize=9)

ax2.scatter(
    sharpe_results[1, sharpe_max_sharpe_idx],
    sharpe_results[0, sharpe_max_sharpe_idx],
    marker="*",
    color="#dc2626",
    s=300,
    edgecolors="white",
    linewidths=1.5,
    label="Max Sharpe",
    zorder=5,
)
ax2.scatter(
    sharpe_results[1, sharpe_min_vol_idx],
    sharpe_results[0, sharpe_min_vol_idx],
    marker="D",
    color="#059669",
    s=100,
    edgecolors="white",
    linewidths=1.5,
    label="GMV",
    zorder=5,
)

ax2.set_xlabel("Annualized Volatility", fontweight="medium")
ax2.set_ylabel("Annualized Return", fontweight="medium")
ax2.set_title("Efficient Frontier: Max Sharpe Selection", fontweight="600", pad=12)
ax2.legend(loc="upper right", frameon=True)
ax2.grid(True, linestyle="--", alpha=0.5, color="#94a3b8")
ax2.set_axisbelow(True)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

fig1.tight_layout()
fig1.savefig("efficient_frontier_comparison.png", dpi=150, bbox_inches="tight")
plt.close(fig1)

# Figure 2: Correlation Heatmaps
fig2, (ax3, ax4) = plt.subplots(1, 2, figsize=(18, 8), facecolor="#fafafa")

# HRP Correlation
ax3.set_facecolor("white")
sns.heatmap(
    hrp_returns.corr(),
    annot=True,
    cmap="RdBu_r",
    center=0,
    fmt=".2f",
    square=True,
    linewidths=0.5,
    linecolor="white",
    annot_kws={"size": 9, "weight": "medium"},
    cbar_kws={"shrink": 0.8, "label": "Correlation"},
    ax=ax3,
    vmin=-1,
    vmax=1,
)
ax3.set_title("Correlation: HRP Selection", fontweight="600", pad=12)
ax3.tick_params(axis="both", labelsize=9)

# Max Sharpe Correlation
ax4.set_facecolor("white")
sns.heatmap(
    sharpe_returns.corr(),
    annot=True,
    cmap="RdBu_r",
    center=0,
    fmt=".2f",
    square=True,
    linewidths=0.5,
    linecolor="white",
    annot_kws={"size": 9, "weight": "medium"},
    cbar_kws={"shrink": 0.8, "label": "Correlation"},
    ax=ax4,
    vmin=-1,
    vmax=1,
)
ax4.set_title("Correlation: Max Sharpe Selection", fontweight="600", pad=12)
ax4.tick_params(axis="both", labelsize=9)

fig2.tight_layout()
fig2.savefig("correlation_comparison.png", dpi=150, bbox_inches="tight")
plt.close(fig2)

print("✓ Visualizations saved: efficient_frontier_comparison.png, correlation_comparison.png")


Data loaded: 739 days total
Date range: 2023-02-02 to 2026-01-30

Average correlation for each stock (with all other stocks):
HINDUNILVR    0.129338
DMART         0.137562
CIPLA         0.160962
ITC           0.183127
NESTLEIND     0.190527
DRREDDY       0.194073
TCS           0.207814
INFY          0.208898
SUNPHARMA     0.209995
TRENT         0.212914
INDUSTOWER    0.217787
BAJAJ-AUTO    0.223687
HCLTECH       0.225634
TITAN         0.226862
HDFCBANK      0.227458
ICICIBANK     0.236895
MARUTI        0.238012
BHARTIARTL    0.242085
TATACOMM      0.246117
POWERGRID     0.266451
HINDALCO      0.270862
M&M           0.274957
ADANIPORTS    0.277158
SBIN          0.278540
ULTRACEMCO    0.286209
LT            0.286585
NTPC          0.295564
RELIANCE      0.311088
JSWSTEEL      0.327269
TATASTEEL     0.329103

METHOD 1: Hierarchical Risk Parity (HRP) Stock Selection
Automobiles: BAJAJ-AUTO (avg distance: 0.5647)
Banking & Finance: SBIN (avg distance: 0.5501)
Construction/Infra: ULTRACEMCO (

In [2]:
# Print clear summary of Sharpe Ratios for both methods
print("\n" + "=" * 70)
print("SHARPE RATIO SUMMARY - HRP vs MAX SHARPE SELECTION")
print("=" * 70)
print("\nHierarchical Risk Parity (HRP) Method:")
print(f"  HRP - GMV Portfolio:        Sharpe = {hrp_results[2, hrp_min_vol_idx]:.4f}")
print(f"  HRP - Max Sharpe Portfolio: Sharpe = {hrp_results[2, hrp_max_sharpe_idx]:.4f}")

print("\nMax Sharpe Ratio Selection Method:")
print(f"  Sharpe - GMV Portfolio:        Sharpe = {sharpe_results[2, sharpe_min_vol_idx]:.4f}")
print(f"  Sharpe - Max Sharpe Portfolio: Sharpe = {sharpe_results[2, sharpe_max_sharpe_idx]:.4f}")

print("\n" + "-" * 70)
print("WINNER:")
all_sharpes = {
    "HRP - GMV": hrp_results[2, hrp_min_vol_idx],
    "HRP - Max Sharpe": hrp_results[2, hrp_max_sharpe_idx],
    "Sharpe - GMV": sharpe_results[2, sharpe_min_vol_idx],
    "Sharpe - Max Sharpe": sharpe_results[2, sharpe_max_sharpe_idx]
}
best_portfolio = max(all_sharpes, key=all_sharpes.get)
print(f"Best Portfolio: {best_portfolio} with Sharpe Ratio = {all_sharpes[best_portfolio]:.4f}")
print("=" * 70)



SHARPE RATIO SUMMARY - HRP vs MAX SHARPE SELECTION

Hierarchical Risk Parity (HRP) Method:
  HRP - GMV Portfolio:        Sharpe = 1.2058
  HRP - Max Sharpe Portfolio: Sharpe = 1.7668

Max Sharpe Ratio Selection Method:
  Sharpe - GMV Portfolio:        Sharpe = 1.4842
  Sharpe - Max Sharpe Portfolio: Sharpe = 1.7995

----------------------------------------------------------------------
WINNER:
Best Portfolio: Sharpe - Max Sharpe with Sharpe Ratio = 1.7995


In [3]:
# Custom stock selection - User-specified stocks
custom_stocks = ['ICICIBANK', 'HCLTECH', 'BAJAJ-AUTO', 'NESTLEIND', 'NTPC', 
                 'SUNPHARMA', 'HINDALCO', 'ADANIPORTS', 'BHARTIARTL', 'TRENT']

print("=" * 70)
print("CUSTOM STOCK SELECTION - PORTFOLIO OPTIMIZATION")
print("=" * 70)
print(f"\nSelected Stocks: {', '.join(custom_stocks)}")
print()

# Run optimization on custom stocks
custom_results, custom_returns, custom_max_sharpe_idx, custom_min_vol_idx, custom_gmv_w, custom_sharpe_w = optimize_portfolio(
    custom_stocks, returns_df, "Custom Stock Selection"
)

# Detailed summary
print("\n" + "=" * 70)
print("CUSTOM STOCKS - SHARPE RATIO SUMMARY")
print("=" * 70)
print(f"\nGMV Portfolio (Minimum Variance):")
print(f"  Return:     {custom_results[0, custom_min_vol_idx]:.2%}")
print(f"  Volatility: {custom_results[1, custom_min_vol_idx]:.2%}")
print(f"  Sharpe:     {custom_results[2, custom_min_vol_idx]:.4f}")

print(f"\nMax Sharpe Portfolio:")
print(f"  Return:     {custom_results[0, custom_max_sharpe_idx]:.2%}")
print(f"  Volatility: {custom_results[1, custom_max_sharpe_idx]:.2%}")
print(f"  Sharpe:     {custom_results[2, custom_max_sharpe_idx]:.4f}")

print("\n" + "=" * 70)
print("COMPARISON WITH OTHER METHODS")
print("=" * 70)
all_methods_comparison = pd.DataFrame({
    "Method": [
        "HRP - GMV", 
        "HRP - Max Sharpe", 
        "Sharpe - GMV", 
        "Sharpe - Max Sharpe",
        "CUSTOM - GMV",
        "CUSTOM - Max Sharpe"
    ],
    "Return": [
        hrp_results[0, hrp_min_vol_idx],
        hrp_results[0, hrp_max_sharpe_idx],
        sharpe_results[0, sharpe_min_vol_idx],
        sharpe_results[0, sharpe_max_sharpe_idx],
        custom_results[0, custom_min_vol_idx],
        custom_results[0, custom_max_sharpe_idx]
    ],
    "Volatility": [
        hrp_results[1, hrp_min_vol_idx],
        hrp_results[1, hrp_max_sharpe_idx],
        sharpe_results[1, sharpe_min_vol_idx],
        sharpe_results[1, sharpe_max_sharpe_idx],
        custom_results[1, custom_min_vol_idx],
        custom_results[1, custom_max_sharpe_idx]
    ],
    "Sharpe": [
        hrp_results[2, hrp_min_vol_idx],
        hrp_results[2, hrp_max_sharpe_idx],
        sharpe_results[2, sharpe_min_vol_idx],
        sharpe_results[2, sharpe_max_sharpe_idx],
        custom_results[2, custom_min_vol_idx],
        custom_results[2, custom_max_sharpe_idx]
    ]
})
all_methods_comparison["Return"] = all_methods_comparison["Return"].map(lambda x: f"{x:.2%}")
all_methods_comparison["Volatility"] = all_methods_comparison["Volatility"].map(lambda x: f"{x:.2%}")
all_methods_comparison["Sharpe"] = all_methods_comparison["Sharpe"].map(lambda x: f"{x:.4f}")
print(all_methods_comparison.to_string(index=False))
print()

# Highlight the winner
all_sharpe_values = {
    "HRP - GMV": hrp_results[2, hrp_min_vol_idx],
    "HRP - Max Sharpe": hrp_results[2, hrp_max_sharpe_idx],
    "Sharpe - GMV": sharpe_results[2, sharpe_min_vol_idx],
    "Sharpe - Max Sharpe": sharpe_results[2, sharpe_max_sharpe_idx],
    "CUSTOM - GMV": custom_results[2, custom_min_vol_idx],
    "CUSTOM - Max Sharpe": custom_results[2, custom_max_sharpe_idx]
}
best = max(all_sharpe_values, key=all_sharpe_values.get)
print(f"🏆 BEST OVERALL: {best} with Sharpe Ratio = {all_sharpe_values[best]:.4f}")
print("=" * 70)


CUSTOM STOCK SELECTION - PORTFOLIO OPTIMIZATION

Selected Stocks: ICICIBANK, HCLTECH, BAJAJ-AUTO, NESTLEIND, NTPC, SUNPHARMA, HINDALCO, ADANIPORTS, BHARTIARTL, TRENT


PORTFOLIO OPTIMIZATION: Custom Stock Selection

Weights: Global Minimum Variance (GMV) vs Maximum Sharpe
     Stock GMV weight Max Sharpe weight
 ICICIBANK     20.43%             0.69%
   HCLTECH      5.73%            12.56%
BAJAJ-AUTO      2.44%            16.66%
 NESTLEIND     24.31%             5.29%
      NTPC      7.35%             7.46%
 SUNPHARMA     21.62%             6.39%
  HINDALCO      0.10%             2.05%
ADANIPORTS      0.52%            10.12%
BHARTIARTL     11.56%            28.48%
     TRENT      5.93%            10.30%

Return, Volatility, and Sharpe Ratio (annualized):
         Portfolio Return (ann.) Volatility (ann.) Sharpe Ratio
GMV (min variance)        22.43%            11.62%       1.3713
        Max Sharpe        32.53%            14.59%       1.7845


CUSTOM STOCKS - SHARPE RATIO SUMMARY

GMV

In [4]:
# PRINT ALL SHARPE RATIOS AT ONCE
print("\n" + "█" * 70)
print("█" + " " * 68 + "█")
print("█" + " " * 20 + "ALL SHARPE RATIOS" + " " * 31 + "█")
print("█" + " " * 68 + "█")
print("█" * 70)

print("\n📊 METHOD 1: Hierarchical Risk Parity (HRP)")
print(f"   • HRP - GMV Portfolio:        {hrp_results[2, hrp_min_vol_idx]:.4f}")
print(f"   • HRP - Max Sharpe Portfolio: {hrp_results[2, hrp_max_sharpe_idx]:.4f}")

print("\n📊 METHOD 2: Max Sharpe Ratio Selection")
print(f"   • Sharpe - GMV Portfolio:        {sharpe_results[2, sharpe_min_vol_idx]:.4f}")
print(f"   • Sharpe - Max Sharpe Portfolio: {sharpe_results[2, sharpe_max_sharpe_idx]:.4f}")

print("\n📊 METHOD 3: Custom Stock Selection")
print(f"   • Custom - GMV Portfolio:        {custom_results[2, custom_min_vol_idx]:.4f}")
print(f"   • Custom - Max Sharpe Portfolio: {custom_results[2, custom_max_sharpe_idx]:.4f}")

print("\n" + "─" * 70)
# Find the best
all_sharpes_dict = {
    "HRP - GMV": hrp_results[2, hrp_min_vol_idx],
    "HRP - Max Sharpe": hrp_results[2, hrp_max_sharpe_idx],
    "Sharpe - GMV": sharpe_results[2, sharpe_min_vol_idx],
    "Sharpe - Max Sharpe": sharpe_results[2, sharpe_max_sharpe_idx],
    "Custom - GMV": custom_results[2, custom_min_vol_idx],
    "Custom - Max Sharpe": custom_results[2, custom_max_sharpe_idx]
}

best_method = max(all_sharpes_dict, key=all_sharpes_dict.get)
worst_method = min(all_sharpes_dict, key=all_sharpes_dict.get)

print(f"🏆 HIGHEST: {best_method} = {all_sharpes_dict[best_method]:.4f}")
print(f"📉 LOWEST:  {worst_method} = {all_sharpes_dict[worst_method]:.4f}")
print(f"📈 RANGE:   {all_sharpes_dict[best_method] - all_sharpes_dict[worst_method]:.4f}")
print("█" * 70)



██████████████████████████████████████████████████████████████████████
█                                                                    █
█                    ALL SHARPE RATIOS                               █
█                                                                    █
██████████████████████████████████████████████████████████████████████

📊 METHOD 1: Hierarchical Risk Parity (HRP)
   • HRP - GMV Portfolio:        1.2058
   • HRP - Max Sharpe Portfolio: 1.7668

📊 METHOD 2: Max Sharpe Ratio Selection
   • Sharpe - GMV Portfolio:        1.4842
   • Sharpe - Max Sharpe Portfolio: 1.7995

📊 METHOD 3: Custom Stock Selection
   • Custom - GMV Portfolio:        1.3713
   • Custom - Max Sharpe Portfolio: 1.7845

──────────────────────────────────────────────────────────────────────
🏆 HIGHEST: Sharpe - Max Sharpe = 1.7995
📉 LOWEST:  HRP - GMV = 1.2058
📈 RANGE:   0.5937
██████████████████████████████████████████████████████████████████████


In [5]:
# ============================================================
# COMPREHENSIVE WEIGHTS SUMMARY - ALL METHODS
# ============================================================
print("\n" + "█" * 90)
print("█" + " " * 88 + "█")
print("█" + " " * 25 + "PORTFOLIO WEIGHTS SUMMARY (3-YEAR DATA)" + " " * 24 + "█")
print("█" + " " * 88 + "█")
print("█" * 90)

# ============================================================
# METHOD 1: HRP Selection Weights
# ============================================================
print("\n" + "=" * 90)
print("METHOD 1: HIERARCHICAL RISK PARITY (HRP) - PORTFOLIO WEIGHTS")
print("=" * 90)

hrp_weights_df = pd.DataFrame({
    "Stock": hrp_stocks,
    "GMV Weight": hrp_gmv_w,
    "Max Sharpe Weight": hrp_sharpe_w
})
hrp_weights_df["GMV Weight"] = hrp_weights_df["GMV Weight"].map(lambda x: f"{x:.2%}")
hrp_weights_df["Max Sharpe Weight"] = hrp_weights_df["Max Sharpe Weight"].map(lambda x: f"{x:.2%}")
print(hrp_weights_df.to_string(index=False))

# ============================================================
# METHOD 2: Max Sharpe Selection Weights
# ============================================================
print("\n" + "=" * 90)
print("METHOD 2: MAXIMUM SHARPE RATIO SELECTION - PORTFOLIO WEIGHTS")
print("=" * 90)

sharpe_weights_df = pd.DataFrame({
    "Stock": sharpe_stocks,
    "GMV Weight": sharpe_gmv_w,
    "Max Sharpe Weight": sharpe_sharpe_w
})
sharpe_weights_df["GMV Weight"] = sharpe_weights_df["GMV Weight"].map(lambda x: f"{x:.2%}")
sharpe_weights_df["Max Sharpe Weight"] = sharpe_weights_df["Max Sharpe Weight"].map(lambda x: f"{x:.2%}")
print(sharpe_weights_df.to_string(index=False))

# ============================================================
# METHOD 3: Custom Stock Selection Weights
# ============================================================
print("\n" + "=" * 90)
print("METHOD 3: CUSTOM STOCK SELECTION - PORTFOLIO WEIGHTS")
print("=" * 90)

custom_weights_df = pd.DataFrame({
    "Stock": custom_stocks,
    "GMV Weight": custom_gmv_w,
    "Max Sharpe Weight": custom_sharpe_w
})
custom_weights_df["GMV Weight"] = custom_weights_df["GMV Weight"].map(lambda x: f"{x:.2%}")
custom_weights_df["Max Sharpe Weight"] = custom_weights_df["Max Sharpe Weight"].map(lambda x: f"{x:.2%}")
print(custom_weights_df.to_string(index=False))

# ============================================================
# Stock Selection Comparison
# ============================================================
print("\n" + "=" * 90)
print("STOCK SELECTIONS BY METHOD")
print("=" * 90)
print("\nHRP Stocks:")
print(", ".join(hrp_stocks))
print("\nMax Sharpe Stocks:")
print(", ".join(sharpe_stocks))
print("\nCustom Stocks:")
print(", ".join(custom_stocks))

# Find top weighted stocks in each portfolio
print("\n" + "=" * 90)
print("TOP 3 WEIGHTED STOCKS IN EACH PORTFOLIO")
print("=" * 90)

def get_top_3(stocks, weights, portfolio_name):
    w_df = pd.DataFrame({"Stock": stocks, "Weight": weights})
    w_df = w_df.sort_values("Weight", ascending=False).head(3)
    print(f"\n{portfolio_name}:")
    for _, row in w_df.iterrows():
        print(f"  {row['Stock']}: {row['Weight']:.2%}")

get_top_3(hrp_stocks, hrp_gmv_w, "HRP - GMV")
get_top_3(hrp_stocks, hrp_sharpe_w, "HRP - Max Sharpe")
get_top_3(sharpe_stocks, sharpe_gmv_w, "Sharpe Selection - GMV")
get_top_3(sharpe_stocks, sharpe_sharpe_w, "Sharpe Selection - Max Sharpe")
get_top_3(custom_stocks, custom_gmv_w, "Custom - GMV")
get_top_3(custom_stocks, custom_sharpe_w, "Custom - Max Sharpe")

print("\n" + "█" * 90)



██████████████████████████████████████████████████████████████████████████████████████████
█                                                                                        █
█                         PORTFOLIO WEIGHTS SUMMARY (3-YEAR DATA)                        █
█                                                                                        █
██████████████████████████████████████████████████████████████████████████████████████████

METHOD 1: HIERARCHICAL RISK PARITY (HRP) - PORTFOLIO WEIGHTS
     Stock GMV Weight Max Sharpe Weight
BAJAJ-AUTO      9.10%            28.80%
      SBIN      8.49%            17.15%
ULTRACEMCO      8.16%             5.08%
       ITC     22.23%             0.77%
  RELIANCE      1.17%             0.64%
   HCLTECH     10.72%             1.22%
  HINDALCO      6.37%             5.67%
 SUNPHARMA     21.53%             8.47%
     TRENT      2.31%             9.05%
BHARTIARTL      9.91%            23.14%

METHOD 2: MAXIMUM SHARPE RATIO SELECTION 